## Random Forest Baseline Model

This notebook trains a baseline Random Forest classifier from the Stage-03 feature bundle. A curated event label table is required.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd() / "src", Path.cwd().parent / "src"]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))


In [ ]:
from pathlib import Path
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
X = np.load("ml_data/rf_X.npy")
y = np.load("ml_data/rf_y.npy")
mask = ~np.isnan(y)
X = X[mask]
y = y[mask].astype(int)
X.shape, y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y if len(np.unique(y)) > 1 else None
)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, class_weight="balanced")),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump(pipeline, "models/random_forest_baseline.joblib")
"models/random_forest_baseline.joblib"